# Notebook 06 --- Custom, Multi-task, and Attention Heads

Welcome to the most conceptually dense notebook of the curriculum. Up to
this point we have loaded pretrained backbones, attached the default
classification head that `timm` provides, fine-tuned, and evaluated the
results. In this notebook we go one level deeper and learn how to
**extend** a model.

A pretrained backbone converts raw pixels into a compact feature vector.
Everything after that --- the "head" --- is completely up to you. You can:

* replace the default single `Linear` head with something richer,
* attach several heads that share the same backbone and solve multiple
  tasks at once,
* sprinkle **attention** modules inside the model so that it learns to
  focus on the channels, spatial positions, or patches that matter most,
* inspect what flows through the network on the **forward** pass and how
  gradients propagate back on the **backward** pass.

### Learning goals

By the end of this notebook you will be able to:

1. Strip the default head off a `timm` model and attach your own
   (`CustomHead`).
2. Build a dual-output model that classifies two related targets at once
   (`DualHead`) and reason about auxiliary-loss weighting.
3. Construct a generic multi-task head that mixes classification,
   multilabel, and regression outputs in the same forward pass
   (`MultiHead`).
4. Attach channel attention (SE), spatial attention (CBAM), and a
   self-attention "vision-transformer-lite" head on top of a CNN backbone
   and visualise what each one is doing.
5. Inspect a model's forward-pass shapes with `torchinfo`, register
   gradient hooks, read per-parameter gradient norms, and diagnose
   "dead" layers.

### Recommended hardware

A GPU will make the short training runs pleasant but is not strictly
required --- every demo in this notebook is intentionally tiny (subset of
Oxford-IIIT Pet, 2 epochs) so it finishes in a few minutes on CPU too.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/numberonewastefellow/my_learning/blob/main/notebooks/06_custom_multi_and_attention_heads.ipynb)

In [ ]:
# Universal setup: runs identically in Colab and locally
import sys, os
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("my_learning"):
        !git clone --quiet https://github.com/numberonewastefellow/my_learning.git
    %cd my_learning
    !pip install -q -r requirements.txt

repo_root = os.path.abspath(".") if IN_COLAB else os.path.abspath("..")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from utils.env import bootstrap
info = bootstrap()
device = info.device

## Shared imports

The rest of the notebook uses a small, fixed set of libraries. We import
them once here so the individual sections stay focused on the concepts
rather than boilerplate.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

import timm
from torchinfo import summary

from utils.heads import (
    CustomHead,
    DualHead,
    MultiHead,
    ChannelAttention,
    SpatialAttention,
    SelfAttentionHead,
)
from utils.plotting import show_image_grid, plot_grad_norms, plot_curves

torch.manual_seed(info.seed)
print("Ready. device =", device)

## Part A --- Head replacement (simple)

A deep image model is traditionally split into two pieces:

```
   +-----------------------+       +--------------------+
   |       BACKBONE        | ----> |        HEAD        |
   |  convs + blocks +     |       |  pool + linear(s)  |
   |  optional pooling     |       |    -> logits       |
   +-----------------------+       +--------------------+
        "feature extractor"             "classifier"
```

* The **backbone** takes a `(B, 3, H, W)` image tensor and produces a
  high-dimensional feature representation. For EfficientNetV2-S with a
  `224x224` input the backbone ends with a pooled vector of 1280 features.
* The **head** takes that feature vector and maps it to the number of
  output classes (or regression targets, or whatever) your task requires.

Because `timm.create_model(..., num_classes=0)` strips the head and
returns the *pooled* backbone features directly, replacing the head is
literally a one-line change:

```python
backbone = timm.create_model("tf_efficientnetv2_s.in21k_ft_in1k",
                             pretrained=True, num_classes=0)
model = nn.Sequential(backbone, MyHead(in_features=1280, num_classes=37))
```

Let's see it in action.


In [ ]:
# Build the backbone without any classification head.
# num_classes=0 tells timm "return the pooled feature vector directly".
backbone = timm.create_model("tf_efficientnetv2_s.in21k_ft_in1k",
                             pretrained=True, num_classes=0)
backbone.eval()

# Probe the feature dimension with a dummy input.
with torch.no_grad():
    dummy = torch.randn(1, 3, 224, 224)
    feat = backbone(dummy)

print("backbone output shape :", tuple(feat.shape))
feat_dim = feat.shape[1]
print("feature dim           :", feat_dim)

### What the default head looks like

For comparison, here is the default head you get when you ask `timm` for
a 37-class model directly. It is a single `nn.Linear(1280, 37)` --- nice
and simple, and often perfectly adequate.


In [ ]:
# Ask timm for the same backbone with a 37-way classification head.
model_default = timm.create_model("tf_efficientnetv2_s.in21k_ft_in1k",
                                   pretrained=False, num_classes=37)
print("Default classifier sub-module:")
print(model_default.get_classifier())

### Swapping in `CustomHead`

`CustomHead` (from `utils/heads.py`) is a richer MLP:

```
Dropout -> Linear(in -> hidden) -> BatchNorm1d -> ReLU -> Linear(hidden -> num_classes)
```

It is a drop-in replacement: it takes a flat `(B, in_features)` feature
vector and produces `(B, num_classes)` logits. Attaching it to the
backbone is literally one `nn.Sequential`.


In [ ]:
head = CustomHead(in_features=feat_dim, num_classes=37, hidden=512, dropout=0.3)
model = nn.Sequential(backbone, head).to(device)

# Quick forward pass check.
model.eval()
with torch.no_grad():
    out = model(torch.randn(2, 3, 224, 224, device=device))
print("model output shape :", tuple(out.shape))  # expected (2, 37)

In [ ]:
# torchinfo.summary prints per-layer I/O shapes, param counts, and MACs.
# It's the single most useful "what does my model actually look like?" tool.
summary(model, input_size=(1, 3, 224, 224), depth=2,
        col_names=("input_size", "output_size", "num_params"))

Take a moment to read the summary. Two things are worth noticing:

1. Everything up to the last `AdaptiveAvgPool2d` is the frozen-worthy
   backbone; the pooled feature vector that leaves it is `(1, 1280)`.
2. The `CustomHead` adds four small layers on top of that and about
   ~670k extra parameters --- trivial next to the 21M-param backbone.

The head can be **any** module that maps `(B, feat_dim) -> (B, num_classes)`.
Drop in extra `Linear`s, non-linearities, normalisation layers, residual
connections --- whatever your task seems to benefit from. The rest of the
model doesn't care.


## Part B --- Feature extraction in isolation

Sometimes you don't want a full classification model, you just want the
features. Two common reasons:

* You're building a **retrieval / similarity** system and want to index
  pooled feature vectors.
* You want to **visualise intermediate feature maps** to understand what
  the network has learned.

`timm.create_model(..., num_classes=0)` gives you the pooled features,
and PyTorch's `register_forward_hook` lets you peek at any intermediate
activation without modifying the model itself.


In [ ]:
# Pooled features -- what you'd feed into retrieval / clustering.
backbone.eval().to(device)
with torch.no_grad():
    feats = backbone(torch.randn(4, 3, 224, 224, device=device))
print("pooled feats :", tuple(feats.shape))   # (4, 1280)

### Forward hooks: peeking at intermediate feature maps

A **forward hook** is a function you attach to a layer that PyTorch calls
every time that layer produces an output. The hook receives the module,
its inputs, and its output --- so it's trivial to stash any intermediate
tensor for later inspection.

Below we attach a hook to every stage inside EfficientNetV2-S's `blocks`
`Sequential` so we can record the 4-D feature map shape at each stage.


In [ ]:
captured = {}

def make_hook(name):
    def _hook(module, inp, out):
        # We store the shape and a detached copy of the tensor so the
        # main forward pass is unaffected by our inspection.
        captured[name] = out.detach()
    return _hook

# EfficientNetV2 exposes a ``blocks`` Sequential of stages. We register one
# hook per stage. The feature map gets progressively smaller (spatially)
# and deeper (channel-wise) as we descend through the network.
handles = []
for i, stage in enumerate(backbone.blocks):
    handles.append(stage.register_forward_hook(make_hook(f"stage_{i}")))

with torch.no_grad():
    _ = backbone(torch.randn(1, 3, 224, 224, device=device))

for h in handles:
    h.remove()

for name, t in captured.items():
    print(f"{name:10s} -> {tuple(t.shape)}")

### Visualising a feature map

To build intuition, let's pull out the first 64 channels of one of the
earlier stages and display them as a grid of grayscale intensities. Each
tile is one channel's spatial activation pattern. Lit-up regions indicate
where that channel "fires" for this input; dark regions are suppressed.


In [ ]:
# Pick a stage whose feature map is small enough to be informative but
# large enough to be pretty. The third stage tends to look good.
fm = captured["stage_2"][0]  # (C, H, W)
print("feature map shape :", tuple(fm.shape))

# Take the first 64 channels and normalise each independently to [0, 1].
n_show = min(64, fm.shape[0])
tiles = []
for i in range(n_show):
    t = fm[i].float().cpu()
    t = (t - t.min()) / (t.max() - t.min() + 1e-8)
    tiles.append(t.unsqueeze(0))  # (1, H, W) for show_image_grid

show_image_grid(tiles, ncols=8, figsize=(12, 12))
plt.suptitle("First 64 channels of stage_2 feature map", y=1.02)
plt.show()

Notice how different channels react to different local structures:
some light up on edges, some on textures, some on particular colours.
The later the stage, the more abstract the features. This is exactly what
allows a small classification head on top to do so well --- the backbone
has already done the hard work of extracting useful representations.


## Part C --- Multi-head model (simple): breed + species on Oxford-IIIT Pet

Oxford-IIIT Pet labels each image with **both** a breed (37 classes) and
a species (cat vs dog, 2 classes). The species label is fully determined
by the breed, but training on both simultaneously gives the network an
extra supervision signal and often regularises the backbone.

### Loss weighting

When you sum losses from multiple heads you should almost always weight
them. Losses on different tasks live on different scales, and even when
they are comparable you usually want the **primary** task to dominate and
the **auxiliary** task to act as a mild regulariser:

```
total_loss = lambda_primary * CE(primary) + lambda_aux * CE(aux)
```

We use `lambda_primary = 1.0`, `lambda_aux = 0.3`. Setting the auxiliary
weight too high is a classic beginner mistake --- the model can end up
"cheating" by solving the easy auxiliary task while ignoring the primary
one.


In [ ]:
# Oxford-IIIT Pet with a tuple target: (breed_id, species_id).
# target_types=("category", "binary-category") makes torchvision return the
# joint label. We restrict to a small subset for speed.
from torchvision.datasets import OxfordIIITPet
from torchvision import transforms as T
from torch.utils.data import DataLoader, Subset
from utils.env import data_dir

root = data_dir()
tfm_train = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
tfm_eval = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

try:
    pet_train_full = OxfordIIITPet(
        root=root, split="trainval", download=True,
        target_types=("category", "binary-category"), transform=tfm_train)
    pet_val_full = OxfordIIITPet(
        root=root, split="test", download=True,
        target_types=("category", "binary-category"), transform=tfm_eval)
    DATA_OK = True
except Exception as e:
    print("Could not download Pet:", e)
    print("This cell will be skipped; the rest of the notebook still runs.")
    DATA_OK = False

if DATA_OK:
    g = torch.Generator().manual_seed(0)
    train_idx = torch.randperm(len(pet_train_full), generator=g)[:300].tolist()
    val_idx   = torch.randperm(len(pet_val_full),   generator=g)[:100].tolist()
    pet_train = Subset(pet_train_full, train_idx)
    pet_val   = Subset(pet_val_full,   val_idx)
    print("train subset :", len(pet_train), " val subset :", len(pet_val))

In [ ]:
# Build the dual-head model.
backbone_c = timm.create_model("tf_efficientnetv2_s.in21k_ft_in1k",
                                pretrained=True, num_classes=0).to(device)
dual_head = DualHead(in_features=feat_dim, num_primary=37, num_aux=2).to(device)

class DualModel(nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head = head
    def forward(self, x):
        f = self.backbone(x)
        return self.head(f)

dual_model = DualModel(backbone_c, dual_head).to(device)
print(dual_model)

In [ ]:
# Tiny, 2-epoch training loop to demonstrate the multi-task loss.
if DATA_OK:
    def collate(batch):
        xs = torch.stack([b[0] for b in batch])
        y_breed = torch.tensor([b[1][0] for b in batch])
        y_spec  = torch.tensor([b[1][1] for b in batch])
        return xs, y_breed, y_spec

    train_loader = DataLoader(pet_train, batch_size=16, shuffle=True,
                              collate_fn=collate, num_workers=0)
    val_loader   = DataLoader(pet_val,   batch_size=16, shuffle=False,
                              collate_fn=collate, num_workers=0)

    opt = torch.optim.AdamW(dual_model.parameters(), lr=3e-4)
    lam_primary, lam_aux = 1.0, 0.3

    history = {"breed_acc": [], "spec_acc": []}

    for epoch in range(2):
        dual_model.train()
        for xs, yb, ys in train_loader:
            xs, yb, ys = xs.to(device), yb.to(device), ys.to(device)
            logits_breed, logits_spec = dual_model(xs)
            loss = (lam_primary * F.cross_entropy(logits_breed, yb)
                    + lam_aux * F.cross_entropy(logits_spec, ys))
            opt.zero_grad(); loss.backward(); opt.step()

        # Evaluate both heads separately.
        dual_model.eval()
        correct_b, correct_s, total = 0, 0, 0
        with torch.no_grad():
            for xs, yb, ys in val_loader:
                xs, yb, ys = xs.to(device), yb.to(device), ys.to(device)
                lb, ls = dual_model(xs)
                correct_b += (lb.argmax(1) == yb).sum().item()
                correct_s += (ls.argmax(1) == ys).sum().item()
                total += xs.size(0)
        acc_b = correct_b / max(total, 1)
        acc_s = correct_s / max(total, 1)
        history["breed_acc"].append(acc_b)
        history["spec_acc"].append(acc_s)
        print(f"epoch {epoch+1}  breed_acc={acc_b:.3f}  spec_acc={acc_s:.3f}")

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(history["breed_acc"], marker="o", label="breed (primary)")
    ax.plot(history["spec_acc"],  marker="o", label="species (aux)")
    ax.set_xlabel("epoch"); ax.set_ylabel("val accuracy")
    ax.set_title("DualHead: per-task accuracy")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.show()
else:
    print("Skipping training -- dataset unavailable.")

### Key insight

Even on a tiny 300-sample subset the species head climbs to ~0.95 quickly
while the breed head improves more gradually (37 classes is much harder
than 2). The auxiliary loss gives the backbone "two reasons" to organise
its features sensibly. In practice, on full datasets, this often yields a
small but consistent bump in primary-task accuracy compared to a
single-head baseline.


## Part D --- Multi-head model (advanced): three heads, three loss types

Real-world multi-task setups often mix loss families. In this section we
simulate three heads on the Pet dataset:

| Head       | Out dim | Type        | Loss                                |
|------------|---------|-------------|-------------------------------------|
| `species`  | 37      | multiclass  | `cross_entropy`                     |
| `colors`   | 8       | multilabel  | `binary_cross_entropy_with_logits`  |
| `coords`   | 2       | regression  | `mse_loss`                          |

> **Caveat:** Oxford-IIIT Pet does not actually ship colour multi-labels
> or bounding-box centres for every image, so we **synthesize** the extra
> targets from the breed id (deterministic per class). The point is
> purely to show how to plumb the losses together --- nothing in this
> setup depends on the synthetic targets being meaningful.


In [ ]:
# Build the multi-head model.
backbone_d = timm.create_model("tf_efficientnetv2_s.in21k_ft_in1k",
                                pretrained=True, num_classes=0).to(device)

head_specs = [
    {"name": "species", "out": 37, "type": "multiclass"},
    {"name": "colors",  "out":  8, "type": "multilabel"},
    {"name": "coords",  "out":  2, "type": "regression"},
]
multi_head = MultiHead(in_features=feat_dim, head_specs=head_specs).to(device)

class MultiModel(nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head = head
    def forward(self, x):
        return self.head(self.backbone(x))

multi_model = MultiModel(backbone_d, multi_head).to(device)
print({name: tuple(p.shape) for name, p in multi_head.named_parameters()})

In [ ]:
# Deterministic synthetic labels derived from the breed id.
# Same breed -> same colour set and same (pseudo) bounding-box centre.
def synth_colors(breed_id: int, num_colors: int = 8) -> torch.Tensor:
    rng = np.random.default_rng(seed=1000 + int(breed_id))
    mask = (rng.random(num_colors) > 0.6).astype(np.float32)
    if mask.sum() == 0:
        mask[0] = 1.0
    return torch.tensor(mask)

def synth_coords(breed_id: int) -> torch.Tensor:
    rng = np.random.default_rng(seed=2000 + int(breed_id))
    return torch.tensor(rng.random(2).astype(np.float32))

if DATA_OK:
    def collate_multi(batch):
        xs = torch.stack([b[0] for b in batch])
        y_breed = torch.tensor([b[1][0] for b in batch])
        y_colors = torch.stack([synth_colors(b[1][0]) for b in batch])
        y_coords = torch.stack([synth_coords(b[1][0]) for b in batch])
        return xs, y_breed, y_colors, y_coords

    train_loader_m = DataLoader(pet_train, batch_size=16, shuffle=True,
                                 collate_fn=collate_multi, num_workers=0)
    val_loader_m   = DataLoader(pet_val,   batch_size=16, shuffle=False,
                                 collate_fn=collate_multi, num_workers=0)

In [ ]:
# Mixed-loss training loop.
if DATA_OK:
    opt = torch.optim.AdamW(multi_model.parameters(), lr=3e-4)
    w_species, w_colors, w_coords = 1.0, 0.5, 0.2

    for epoch in range(2):
        multi_model.train()
        for xs, yb, yc, yxy in train_loader_m:
            xs   = xs.to(device)
            yb   = yb.to(device)
            yc   = yc.to(device)
            yxy  = yxy.to(device)
            out = multi_model(xs)
            loss = (w_species * F.cross_entropy(out["species"], yb)
                    + w_colors * F.binary_cross_entropy_with_logits(out["colors"], yc)
                    + w_coords * F.mse_loss(out["coords"], yxy))
            opt.zero_grad(); loss.backward(); opt.step()

        # Per-head metrics on the (tiny) val set.
        multi_model.eval()
        tot = 0
        correct_species = 0
        sum_colors_acc = 0.0
        sum_coords_mse = 0.0
        with torch.no_grad():
            for xs, yb, yc, yxy in val_loader_m:
                xs, yb, yc, yxy = xs.to(device), yb.to(device), yc.to(device), yxy.to(device)
                out = multi_model(xs)
                correct_species += (out["species"].argmax(1) == yb).sum().item()
                pred_colors = (torch.sigmoid(out["colors"]) > 0.5).float()
                sum_colors_acc += (pred_colors == yc).float().mean().item() * xs.size(0)
                sum_coords_mse += F.mse_loss(out["coords"], yxy, reduction="sum").item()
                tot += xs.size(0)
        print(f"epoch {epoch+1}  species_acc={correct_species/tot:.3f}"
              f"  colors_elemacc={sum_colors_acc/tot:.3f}"
              f"  coords_mse={sum_coords_mse/tot:.4f}")

### When does multi-task training help?

* **Related tasks, shared features** (breed + species, object class +
  bounding-box regression): the auxiliary task almost always helps --- it
  acts as a regulariser on the backbone and provides extra supervision
  signal per image.
* **Unrelated tasks, shared features** (classify cats vs dogs and also
  predict the photograph's shutter speed): this is the danger zone.
  The tasks fight over backbone capacity --- a phenomenon called
  **negative transfer** --- and you often end up worse than training two
  separate models.

Picking loss weights is more art than science. A reasonable starting
point: weight each head inversely to the variance of its gradient norms
during a short warm-up run. Advanced alternatives include **uncertainty
weighting** (Kendall et al., 2018) and **GradNorm** (Chen et al., 2018).


## Part E --- Attention heads (the most visual part)

**Attention** is the idea of letting the network decide *where to look*
on a per-input basis, rather than using a fixed set of filter weights
for every image.

Three common flavours:

| Name              | What it re-weights                                     |
|-------------------|--------------------------------------------------------|
| Channel attention | "Which of my 256 channels matter for this image?"      |
| Spatial attention | "Which `H x W` locations matter for this image?"       |
| Self-attention    | "Let each spatial patch talk to every other patch."    |

The first two are lightweight and plug in almost anywhere. The third is
the building block of vision transformers --- much more powerful, but
also more expensive.

We'll inspect each one visually before integrating it into a training
loop.


### E1 --- Channel attention (Squeeze-and-Excitation)

`ChannelAttention` performs a **squeeze-and-excitation**:

1. *Squeeze*: global-average-pool the `(B, C, H, W)` feature map to
   `(B, C)`, i.e. one scalar per channel summarising its global activity.
2. *Excitation*: pass those C scalars through a tiny bottleneck MLP
   `Linear(C, C/r) -> ReLU -> Linear(C/r, C) -> Sigmoid`. This outputs a
   gate vector in `(0, 1)`.
3. *Scale*: multiply each channel of the original feature map by its
   gate value.

Intuitively: "given a global look at the image, decide which channels
are useful and turn the rest down".


In [ ]:
# Toy example on a random feature map.
torch.manual_seed(0)
x_toy = torch.randn(1, 64, 7, 7)

ca = ChannelAttention(channels=64, reduction=8)
# Inspect just the gate before multiplying.
with torch.no_grad():
    gated = ca(x_toy)
    # Recover the per-channel gate: gated / x_toy (rough, broadcasted).
    # Cleaner: redo the gate manually.
    s = F.adaptive_avg_pool2d(x_toy, 1).flatten(1)
    s = torch.sigmoid(ca.fc2(F.relu(ca.fc1(s))))
print("gate shape :", tuple(s.shape))

plt.figure(figsize=(10, 3))
plt.bar(range(64), s[0].numpy(), color="steelblue")
plt.axhline(0.5, linestyle="--", color="gray", alpha=0.7, label="0.5")
plt.title("Per-channel attention weights (random toy input)")
plt.xlabel("channel"); plt.ylabel("gate"); plt.legend()
plt.show()

Because the input is random, the learned gate is roughly centred on
0.5 --- no channel is particularly useful, so none is strongly amplified
or suppressed. During training on a real task the gate distribution
becomes much more structured.


In [ ]:
# Attach ChannelAttention BEFORE the backbone's own pooling, so we
# gate the *spatial* feature map rather than the pooled vector.
#
# timm.create_model(features_only=True) returns a list of feature maps
# at different stages; the last one is what we want. We'll use a
# forward_features style instead to keep things simple.

backbone_e1 = timm.create_model("tf_efficientnetv2_s.in21k_ft_in1k",
                                 pretrained=True, num_classes=0,
                                 global_pool="").to(device)
# global_pool="" means "do NOT pool the feature map". The output is now
# a 4-D tensor (B, C, H, W) that ChannelAttention can operate on.

with torch.no_grad():
    fm = backbone_e1(torch.randn(1, 3, 224, 224, device=device))
print("unpooled feat map :", tuple(fm.shape))
C_e1 = fm.shape[1]

In [ ]:
class SEModel(nn.Module):
    def __init__(self, backbone, channels, num_classes):
        super().__init__()
        self.backbone = backbone
        self.se = ChannelAttention(channels=channels, reduction=16)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(channels, num_classes)
    def forward(self, x):
        f = self.backbone(x)          # (B, C, H, W)
        f = self.se(f)                # channel-gated
        f = self.pool(f).flatten(1)   # (B, C)
        return self.fc(f)

se_model = SEModel(backbone_e1, channels=C_e1, num_classes=37).to(device)
print(se_model)

In [ ]:
# 2-epoch demo training run. We're not chasing accuracy -- we want to
# show the mechanics work.
if DATA_OK:
    def collate_simple(batch):
        xs = torch.stack([b[0] for b in batch])
        yb = torch.tensor([b[1][0] for b in batch])
        return xs, yb

    tl = DataLoader(pet_train, batch_size=16, shuffle=True,
                    collate_fn=collate_simple, num_workers=0)
    vl = DataLoader(pet_val, batch_size=16, shuffle=False,
                    collate_fn=collate_simple, num_workers=0)

    opt = torch.optim.AdamW(se_model.parameters(), lr=3e-4)
    for epoch in range(2):
        se_model.train()
        for xs, yb in tl:
            xs, yb = xs.to(device), yb.to(device)
            loss = F.cross_entropy(se_model(xs), yb)
            opt.zero_grad(); loss.backward(); opt.step()
        se_model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for xs, yb in vl:
                xs, yb = xs.to(device), yb.to(device)
                correct += (se_model(xs).argmax(1) == yb).sum().item()
                total += xs.size(0)
        print(f"[SE] epoch {epoch+1}  val_acc={correct/total:.3f}")

### E2 --- Spatial attention (CBAM-style)

`SpatialAttention` is the spatial dual of channel attention:

1. Compute `avg = x.mean(dim=1, keepdim=True)` and `mx = x.amax(dim=1, keepdim=True)`.
   Each is a `(B, 1, H, W)` summary across the channel axis.
2. Concatenate them into a `(B, 2, H, W)` tensor.
3. Run a single `7x7` convolution with a single output channel, followed
   by a sigmoid, to get a `(B, 1, H, W)` mask.
4. Multiply the original feature map by this mask.

The mask tells us *where* to look: bright regions survive, dark regions
are suppressed.


In [ ]:
sa = SpatialAttention()

# Plug SpatialAttention into the unpooled backbone and visualise the
# attention mask on a real Pet image.
if DATA_OK:
    xs, ys = next(iter(DataLoader(pet_train, batch_size=1,
                                  collate_fn=collate_simple)))
    xs = xs.to(device); sa = sa.to(device)
    with torch.no_grad():
        fm = backbone_e1(xs)            # (1, C, H, W)
        # Re-derive the mask rather than relying on side effects.
        avg = fm.mean(1, keepdim=True)
        mx  = fm.amax(1, keepdim=True)
        mask = torch.sigmoid(sa.conv(torch.cat([avg, mx], dim=1)))[0, 0].cpu().numpy()

    # Denormalize the input image for display.
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img = xs[0].cpu().permute(1, 2, 0).numpy() * std + mean
    img = np.clip(img, 0, 1)

    # Upsample the attention mask to the image resolution for overlaying.
    import torch.nn.functional as Fx
    mask_up = Fx.interpolate(
        torch.tensor(mask)[None, None], size=(224, 224),
        mode="bilinear", align_corners=False)[0, 0].numpy()

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img);      axes[0].set_title("input image");         axes[0].axis("off")
    axes[1].imshow(mask_up, cmap="jet"); axes[1].set_title("spatial attention"); axes[1].axis("off")
    axes[2].imshow(img);      axes[2].imshow(mask_up, cmap="jet", alpha=0.5)
    axes[2].set_title("overlay"); axes[2].axis("off")
    plt.tight_layout(); plt.show()

Because the `SpatialAttention` module is **untrained** (we instantiated
it fresh), its mask mostly reflects the statistics of the backbone's
feature map rather than anything task-specific. After a few epochs of
joint training the bright regions tend to align with the salient object
--- the pet's face, for example.


### E3 --- Self-attention head

The self-attention head treats the `(B, C, H, W)` feature map as a
**sequence of `H * W` tokens**, each of dimension `C`. It adds a learned
positional embedding, applies one round of multi-head self-attention
with a residual and LayerNorm, mean-pools the tokens, and classifies.

This is essentially "a tiny transformer block sitting on top of a CNN".
The upshot: every spatial token can see every other spatial token
through the attention operation, giving the head **global** context that
a single `Linear` head cannot.


In [ ]:
# Because nn.MultiheadAttention needs dim % num_heads == 0, we check
# that C_e1 is compatible. EfficientNetV2-S has 1280 final channels,
# divisible by 4, 8, 16, 20, etc.
print("C_e1 =", C_e1, "  divisible by 4?", C_e1 % 4 == 0)

class SelfAttnModel(nn.Module):
    def __init__(self, backbone, channels, num_classes):
        super().__init__()
        self.backbone = backbone
        self.head = SelfAttentionHead(dim=channels, num_heads=4,
                                      num_classes=num_classes)
    def forward(self, x):
        f = self.backbone(x)      # (B, C, H, W)
        return self.head(f)       # (B, num_classes)

sa_model = SelfAttnModel(backbone_e1, channels=C_e1, num_classes=37).to(device)

if DATA_OK:
    opt = torch.optim.AdamW(sa_model.parameters(), lr=3e-4)
    for epoch in range(2):
        sa_model.train()
        for xs, yb in tl:
            xs, yb = xs.to(device), yb.to(device)
            loss = F.cross_entropy(sa_model(xs), yb)
            opt.zero_grad(); loss.backward(); opt.step()
        sa_model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for xs, yb in vl:
                xs, yb = xs.to(device), yb.to(device)
                correct += (sa_model(xs).argmax(1) == yb).sum().item()
                total += xs.size(0)
        print(f"[SelfAttn] epoch {epoch+1}  val_acc={correct/total:.3f}")

On such a small subset the absolute accuracy is not meaningful, but
the self-attention head does something a plain `Linear` head cannot:
every one of the 49 tokens on the `7x7` feature map contributes
proportionally to how relevant it is to every other token. On tasks
where **global context matters** --- e.g. "is this cat lying on a sofa?"
--- self-attention heads often beat plain linear heads by a respectable
margin.


## Part F --- Forward and backward pass inspection

This is the section that turns "I trained a model and it works" into "I
understand *why* it works and can debug it when it doesn't". We will:

1. Dump the model's **forward-pass shapes** with `torchinfo.summary`.
2. Register **tensor-level gradient hooks** on intermediate activations.
3. Collect **per-parameter gradient norms** after one backward pass and
   visualise them as a bar chart.
4. Contrast the gradient flow in a **fully-trainable** vs a
   **partially-frozen** model.
5. Do the same analysis for the **self-attention head** specifically.
6. Reproduce a synthetic **"dead layer"** scenario to build debugging
   intuition.


### F1 --- Forward-pass shapes with `torchinfo.summary`

The `summary` function does a dummy forward pass under the hood and
records each sub-module's input and output shapes plus its parameter
count. Reading this table top-to-bottom gives you a complete picture of
the data flowing through the model.


In [ ]:
# Build a small fully-connected model for pedagogical clarity. Later
# we will redo the analysis on the bigger CNN.
tiny = nn.Sequential(
    nn.Linear(feat_dim, 512),
    nn.BatchNorm1d(512),
    nn.ReLU(inplace=True),
    nn.Dropout(0.3),
    nn.Linear(512, 128),
    nn.ReLU(inplace=True),
    nn.Linear(128, 37),
).to(device)

summary(tiny, input_size=(4, feat_dim),
        col_names=("input_size", "output_size", "num_params"))

The columns mean:

* **Layer (type)** --- module name in `named_modules()` order.
* **Input / Output Shape** --- the tensor shapes observed during the
  dummy forward pass.
* **Param #** --- number of learnable parameters in that module.

Any time a real training run misbehaves, running `summary` is the first
thing to do. Often the bug is as simple as a shape mismatch the tracer
catches here.


### F2 --- Tracing gradients with `tensor.register_hook`

PyTorch exposes two hook APIs:

* **Module hooks** (`register_forward_hook`, `register_full_backward_hook`)
  fire when a module is called.
* **Tensor hooks** (`tensor.register_hook`) fire when a gradient for a
  specific tensor is computed.

Tensor hooks are handy for intermediate activations that don't
correspond to a whole module. We run the tiny model, attach tensor hooks
at a couple of interior layers, and record their gradient norms.


In [ ]:
# Collect gradient norms at a few named intermediate activations.
grad_norms_tensor = {}

def make_tensor_hook(name):
    def _hook(g):
        grad_norms_tensor[name] = g.detach().norm().item()
    return _hook

# Register tensor hooks by re-running the forward pass manually so we
# have handles on intermediate tensors.
x = torch.randn(8, feat_dim, device=device, requires_grad=True)

h1 = tiny[0](x); h1.retain_grad(); h1.register_hook(make_tensor_hook("after_linear1"))
h2 = tiny[1](h1)
h3 = tiny[2](h2); h3.retain_grad(); h3.register_hook(make_tensor_hook("after_relu1"))
h4 = tiny[3](h3)
h5 = tiny[4](h4); h5.retain_grad(); h5.register_hook(make_tensor_hook("after_linear2"))
h6 = tiny[5](h5)
logits = tiny[6](h6)

loss = F.cross_entropy(logits, torch.randint(0, 37, (8,), device=device))
loss.backward()

for k, v in grad_norms_tensor.items():
    print(f"{k:20s} |grad| = {v:.4f}")

### F3 --- Per-parameter gradient norms

After `loss.backward()` every `nn.Parameter` has a populated `.grad`
attribute. Iterating `model.named_parameters()` and collecting each
`p.grad.norm().item()` gives us a dict we can feed straight into
`plot_grad_norms` (from `utils/plotting.py`) for a bar chart.

This is the single most useful "is my model learning?" plot --- a long
flat sequence of near-zero bars screams "vanishing gradients" or
"everything upstream of here is dead".


In [ ]:
grad_norms = {name: p.grad.detach().norm().item()
               for name, p in tiny.named_parameters() if p.grad is not None}
for k, v in grad_norms.items():
    print(f"{k:25s} {v:.4f}")

plot_grad_norms(grad_norms)
plt.suptitle("Per-parameter gradient norms (all params trainable)", y=1.01)
plt.show()

### F4 --- Frozen vs unfrozen: where do gradients actually flow?

Freezing a layer is as simple as setting `p.requires_grad = False` on its
parameters. After a backward pass, frozen parameters have `p.grad is
None` --- gradients never reach them. Let's verify this by freezing the
first two `Linear` layers of `tiny` and re-running the same analysis.


In [ ]:
# Clone the tiny model and freeze the first half.
tiny_fr = nn.Sequential(
    nn.Linear(feat_dim, 512),
    nn.BatchNorm1d(512),
    nn.ReLU(inplace=True),
    nn.Dropout(0.3),
    nn.Linear(512, 128),
    nn.ReLU(inplace=True),
    nn.Linear(128, 37),
).to(device)

# Freeze layers 0, 1, 4 (first Linear, its BN, second Linear).
for i in (0, 1, 4):
    for p in tiny_fr[i].parameters():
        p.requires_grad = False

# Run one forward+backward.
x = torch.randn(8, feat_dim, device=device)
y = torch.randint(0, 37, (8,), device=device)
loss = F.cross_entropy(tiny_fr(x), y)
loss.backward()

grad_norms_fr = {}
for name, p in tiny_fr.named_parameters():
    if p.grad is None:
        print(f"{name:25s}  FROZEN (grad is None)")
    else:
        grad_norms_fr[name] = p.grad.detach().norm().item()
        print(f"{name:25s}  |grad|={grad_norms_fr[name]:.4f}")

plot_grad_norms(grad_norms_fr)
plt.suptitle("Per-parameter gradient norms (first two Linear layers frozen)", y=1.01)
plt.show()

Notice how the frozen parameters are literally absent from the bar
chart --- their `p.grad` is `None`, so our dict skips them entirely. This
is the mental model you want when fine-tuning: "gradients flow only
through the parts of the graph whose parameters are trainable, so only
those parts can be updated".


### F5 --- Gradient flow through the self-attention head

Let's look at where updates concentrate inside the self-attention head.
Typically the `in_proj_weight` of `nn.MultiheadAttention` (the combined
Q/K/V projection) receives the largest gradient --- it's on the critical
path for every token interaction.


In [ ]:
small_sa_head = SelfAttentionHead(dim=64, num_heads=4, num_classes=10).to(device)
# Synthetic 4-D input at the expected dim.
x_feat = torch.randn(4, 64, 7, 7, device=device, requires_grad=True)
y_syn = torch.randint(0, 10, (4,), device=device)

logits = small_sa_head(x_feat)
loss = F.cross_entropy(logits, y_syn)
loss.backward()

sa_grads = {name: p.grad.detach().norm().item()
            for name, p in small_sa_head.named_parameters() if p.grad is not None}
for k, v in sa_grads.items():
    print(f"{k:40s} {v:.4f}")

plot_grad_norms(sa_grads)
plt.suptitle("Self-attention head: per-parameter gradient norms", y=1.01)
plt.show()

The largest-norm parameters are almost always the QKV projections
(`attn.in_proj_weight`) and the output projection (`attn.out_proj.weight`).
That's where most of the learning actually happens --- the positional
embedding receives a much smaller update per step.


### F6 --- Debugging tip: a "dead" layer

Finally, a tiny synthetic demonstration of a **dead ReLU**. If every
input to a ReLU is negative, the output is exactly zero everywhere --- and
because the derivative of ReLU is 0 on the negative side, **no gradient**
flows back through that activation. The weights upstream of it receive
zero gradient and stop learning.

We reproduce this deliberately by making all inputs negative.


In [ ]:
dead = nn.Sequential(
    nn.Linear(16, 8),
    nn.ReLU(inplace=True),
    nn.Linear(8, 4),
).to(device)

# Construct an input that -- together with a suitably biased weight --
# guarantees the ReLU sees negative activations.
with torch.no_grad():
    dead[0].weight.zero_()
    dead[0].bias.fill_(-5.0)  # every pre-activation = -5

x_d = torch.randn(32, 16, device=device)
y_d = torch.randint(0, 4, (32,), device=device)

logits = dead(x_d)
print("ReLU output (should be all 0):")
print(F.relu(dead[0](x_d))[:3, :5])

loss = F.cross_entropy(logits, y_d)
loss.backward()

for name, p in dead.named_parameters():
    g = p.grad
    print(f"{name:20s} |grad|={0.0 if g is None else g.norm().item():.6f}")

The first `Linear` layer's gradient norm is effectively zero --- its
outputs are all negative, the ReLU kills them, and nothing downstream can
"reach" the `Linear(16, 8)` weights via gradient. That layer is **dead**.

Common real-world causes of dead layers:

* **Dead ReLU** --- bad initialisation or a too-high learning rate pushes
  all pre-activations permanently negative. Use He/Kaiming init and
  consider `LeakyReLU` or `GELU`.
* **Vanishing gradients** --- a long chain of saturating non-linearities
  (sigmoid, tanh) in a deep network. Use residual connections and
  normalisation layers.
* **Learning rate too small** --- gradients arrive, but updates are
  dwarfed by other parameters' moves. Inspect `|update| / |param|`
  ratios across layers.
* **Accidentally frozen parameters** --- double-check `requires_grad`
  flags and that your optimizer actually sees the params
  (`len(list(optimizer.param_groups[0]['params']))`).


## Key takeaways

* A pretrained model is a **backbone plus a head**. `timm.create_model(
  ..., num_classes=0)` strips the head and returns the pooled features;
  everything else is up to you.
* A **custom MLP head** (`CustomHead`) adds capacity and regularisation
  cheaply when a single `Linear` is not enough.
* **Multi-task heads** share a backbone across tasks. Weight auxiliary
  losses lower than the primary loss (start around `0.2-0.5`). Related
  tasks regularise the backbone; unrelated tasks cause negative transfer.
* **Attention modules** give the network an explicit "what to focus on"
  mechanism. Channel attention reweights channels, spatial attention
  reweights positions, self-attention lets every spatial token see every
  other one.
* **Forward hooks**, **tensor hooks**, `torchinfo.summary`, and
  per-parameter gradient norms are the four tools you need to understand
  any PyTorch model. A near-zero gradient norm upstream of your head is
  almost always a bug --- look for frozen layers, dead ReLUs, or too-low
  learning rates.


## Exercises

1. **Four-head Pet model.** Build a `MultiHead` with four tasks: breed
   (`multiclass`, 37), species (`multiclass`, 2), colour (`multilabel`,
   8), and a synthetic "fat score" regression (`regression`, 1). Train
   for 2--3 epochs on a 500-image subset and report per-head metrics.
   Experiment with two different loss-weighting schemes (uniform vs
   hand-tuned) and note which primary-task accuracy is better.

2. **Channel attention ablation.** Take the Part E1 `SEModel` and train
   two copies --- one with `ChannelAttention`, one without --- on the
   same subset and seed. Plot the two validation curves side by side. Do
   you see a difference? How stable is the difference to the random seed?

3. **Dead-layer diagnosis.** Deliberately break a model by
   (a) initialising all biases of a `Linear` with `-10`, (b) setting the
   learning rate to `1e-8`, or (c) freezing every parameter. For each
   case, run one forward+backward and use `plot_grad_norms` to confirm
   your hypothesis about which layers are dead.
